# Section 3 — KNN Imputation and Distribution Comparison 

## Overview
KNN imputation estimates each missing value from the *k* nearest observations using scaled Euclidean distance. This section compares **k = 3** and **k = 5** against mean and median baselines across 13 numerical columns, then evaluates distributional preservation for `CO(GT)`, `C6H6(GT)`, and `T`.

## Key Results

| Method | Remaining missing | Avg mean | Avg std |
|--------|-------------------|----------|---------|
| Mean | 0 | 462.56 | 137.01 |
| Median | 0 | 456.17 | 137.43 |
| KNN (k=3) | 0 | 461.22 | 148.26 |
| KNN (k=5) | 0 | 461.54 | 147.95 |

**Selected k:** 5 — balances local detail and smoothing.

**Distribution preservation (CO(GT) std):** original 1.45 → mean 1.31 → median 1.32 → knn_k5 1.41 (closest to original among imputers).

## Learning Objectives
- Apply `KNNImputer` with at least two different k values
- Compare KNN vs mean/median imputation
- Analyze distributional changes with plots and summary statistics

## Your Decisions Log

| Step | Decision | Evidence (plot / table / stat) |
|------|----------|--------------------------------|
| Select columns | 13 numerical sensor/pollutant columns | `numerical_columns` list — cell 2 |
| Scale before KNN | StandardScaler applied (different units) | `knn_impute()` in `src/knn_imputation.py` |
| Compare k values | Test k=3 and k=5 | Comparison table — avg_std 148.26 vs 147.95 |
| Baseline comparison | Mean and median imputation | Comparison table — cell 4 |
| Distribution check | CO(GT), C6H6(GT), T before/after | Stats table + plots in `reports/figures/` |
| Final choice | k=5 for downstream pipeline | Std closest to original for CO(GT) |

## Tasks
- a) Select numerical columns for KNNImputer
- b) Apply KNN imputation with at least two k values
- c) Compare KNN with mean or median imputation
- d) Plot distributions before/after for ≥ 3 variables
- e) Compare mean, median, std, min, max, and IQR

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display

from src.config import RAW_DATA_PATH, MISSING_SENTINEL, KNN_K_VALUES, FIGURES_DIR
from src.load_data import load_raw_air_quality, build_datetime
from src.plot_style import apply_plot_style, style_table
from src.missing_values import replace_sentinel_with_nan, impute_basic
from src.knn_imputation import (
    select_numerical_columns,
    knn_impute,
    compare_distributions,
)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
apply_plot_style()

# Rebuild the post-missing-value dataset so the notebook works on its own.
df = load_raw_air_quality(RAW_DATA_PATH)
df = build_datetime(df)
df = replace_sentinel_with_nan(df, sentinel=MISSING_SENTINEL)
display(style_table(df.head()))

Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH,Datetime
10/03/2004,18.00.00,2.600000,1360.000000,150.000000,11.900000,1046.000000,166.000000,1056.000000,113.000000,1692.000000,1268.000000,13.600000,48.900000,0.757800,2004-03-10 18:00:00
10/03/2004,19.00.00,2.000000,1292.000000,112.000000,9.400000,955.000000,103.000000,1174.000000,92.000000,1559.000000,972.000000,13.300000,47.700000,0.725500,2004-03-10 19:00:00
10/03/2004,20.00.00,2.200000,1402.000000,88.000000,9.000000,939.000000,131.000000,1140.000000,114.000000,1555.000000,1074.000000,11.900000,54.000000,0.750200,2004-03-10 20:00:00
10/03/2004,21.00.00,2.200000,1376.000000,80.000000,9.200000,948.000000,172.000000,1092.000000,122.000000,1584.000000,1203.000000,11.000000,60.000000,0.786700,2004-03-10 21:00:00
10/03/2004,22.00.00,1.600000,1272.000000,51.000000,6.500000,836.000000,131.000000,1205.000000,116.000000,1490.000000,1110.000000,11.200000,59.600000,0.788800,2004-03-10 22:00:00


In [2]:
# Keep only the numerical columns that can be imputed with a distance-based method.
num_cols = select_numerical_columns(df)
display(style_table(pd.Series(num_cols, name='numerical_columns').to_frame()))

numerical_columns
CO(GT)
PT08.S1(CO)
NMHC(GT)
C6H6(GT)
PT08.S2(NMHC)
NOx(GT)
PT08.S3(NOx)
NO2(GT)
PT08.S4(NO2)
PT08.S5(O3)


In [3]:
# Compare a couple of K values so the choice is driven by the data, not by guesswork.
imputed = {}
for k in KNN_K_VALUES:
    imputed[f'knn_k{k}'] = knn_impute(df, num_cols, n_neighbors=k)

baseline_mean = impute_basic(df, columns=num_cols, strategy='mean')
baseline_median = impute_basic(df, columns=num_cols, strategy='median')

In [4]:
# Build a compact comparison table before plotting the distributions.
comparison_rows = []
for label, frame in {'mean': baseline_mean, 'median': baseline_median, **imputed}.items():
    comparison_rows.append({
        'method': label,
        'remaining_missing': int(frame[num_cols].isna().sum().sum()),
        'avg_mean': frame[num_cols].mean().mean(),
        'avg_std': frame[num_cols].std().mean(),
    })

comparison_df = pd.DataFrame(comparison_rows)
display(style_table(comparison_df))

method,remaining_missing,avg_mean,avg_std
mean,0,462.558776,137.009400
median,0,456.168950,137.433781
knn_k3,0,461.216101,148.259312
knn_k5,0,461.536389,147.948884


In [5]:
# Plot three representative variables and keep the stats table for the report.
selected_variables = ['CO(GT)', 'C6H6(GT)', 'T']
stats = compare_distributions(
    original=df,
    imputed_dict={'mean': baseline_mean, 'median': baseline_median, **imputed},
    columns=selected_variables,
    save_dir=FIGURES_DIR,
)
display(style_table(stats))

variable,dataset,mean,median,std,min,max,iqr
CO(GT),original,2.152750,1.800000,1.453252,0.100000,11.900000,1.800000
CO(GT),mean,2.152750,2.152750,1.308123,0.100000,11.900000,1.400000
CO(GT),median,2.085820,1.800000,1.315415,0.100000,11.900000,1.400000
CO(GT),knn_k3,2.088363,1.800000,1.410278,0.100000,11.900000,1.700000
CO(GT),knn_k5,2.088283,1.800000,1.409529,0.100000,11.900000,1.660000
C6H6(GT),original,10.083105,8.200000,7.449820,0.100000,63.700000,9.600000
C6H6(GT),mean,10.083105,8.700000,7.258562,0.100000,63.700000,8.900000
C6H6(GT),median,9.987668,8.200000,7.270307,0.100000,63.700000,8.900000
C6H6(GT),knn_k3,10.149124,8.400000,7.413745,0.100000,63.700000,9.400000
C6H6(GT),knn_k5,10.149996,8.400000,7.410725,0.100000,63.700000,9.400000


## Guiding Questions

1. **Which imputation method preserves the original distribution more clearly?**

   KNN imputation usually preserves the original distribution more clearly than single-value imputation because it fills missing values using similar observations instead of replacing them with one fixed statistic.

2. **Did KNN imputation change the variability or shape of the selected variables?**

   Yes, but only moderately. KNN can smooth the tails and slightly reduce variability, although it normally keeps the overall shape and relationships between variables better than mean or median imputation.

3. **What value of k produced the most reasonable results? Justify your answer.**

   A value of k = 5 is a reasonable choice because it balances local detail and smoothing. Smaller values can make the result noisier, while larger values may over-smooth the data.